# `grayt` demo — backward ray tracing around Kerr black holes

A hands-on tour of the public API of **`grayt`** (the Python layer of
`kerr-raytracer`, a replication of *OSIRIS*, arXiv:2202.00086).

Contents:

1. Spacetime basics — `BlackHole`, horizon and ISCO
2. Rendering a Page–Thorne thin-disk image and inspecting the raw per-pixel maps
3. Page–Thorne flux profiles for several spins
4. Single geodesics — `trace`, `orbit_ic`, `plot_orbits_2d`
5. "Photograph" mode — imaging a picture placed behind the hole
6. A 3D scene with traced rays
7. What happens on bad input

Every render below is kept small (≤ 512×256) so the whole notebook runs in
about a minute on a laptop. Figures are also saved to `output/`.

**Requirement:** the compiled core (`make` in the repo root) and
numpy/matplotlib. The package is not installed; we add `python/` to the path
by hand (edit `REPO` if you cloned elsewhere).

In [ ]:
import sys, time
from pathlib import Path

REPO = Path("/Users/blancus/Biblioteca/Papers/Gravitation/kerr-raytracer")
sys.path.insert(0, str(REPO / "python"))

import numpy as np
import matplotlib.pyplot as plt

import grayt

OUT = REPO / "output"
OUT.mkdir(exist_ok=True)
print("grayt version:", grayt.__version__)
print("public API   :", ", ".join(grayt.__all__))

## 1. The spacetime: `grayt.BlackHole`

A frozen dataclass with a single parameter, the dimensionless spin `a`
(geometrized units, $M = 1$). `.horizon` and `.isco` call into the Fortran
core. The `repr` is the plain dataclass one, which is perfectly readable.

In [ ]:
bh = grayt.BlackHole(a=0.9)
print(bh)                       # dataclass repr: BlackHole(a=0.9)
print("outer horizon r_H  =", bh.horizon)
print("prograde ISCO      =", bh.isco)

print()
for a in (0.0, 0.5, 0.9, 0.998):
    b = grayt.BlackHole(a=a)
    print(f"a = {a:5.3f}   r_H = {b.horizon:.4f}   r_ISCO = {b.isco:.4f}")

How discoverable is the main entry point? `help(grayt.render)` gives the
full signature with type hints and defaults, plus a two-line physics summary
citing the equation it implements — short but genuinely useful.

In [ ]:
help(grayt.render)

## 2. Rendering a thin-disk image

Compose three ingredients and render. Conventions to keep in mind (all stated
in the `Camera` docstring — read it, it matters):

* `Camera.theta` is in **degrees** (Boyer–Lindquist observer inclination);
* image-plane extents `x`, `y` are in units of $M$;
* `resolution` is `(nx, ny)` and all returned maps are `(nx, ny)` —
  x first, so transpose before `imshow`.

`ThinDisk(l0=1.8)` sets the constant specific angular momentum of the disk
matter; `r_in=None` defaults to the ISCO of whatever hole you render.

In [ ]:
bh95 = grayt.BlackHole(a=0.95)
cam  = grayt.Camera(r=1000, theta=85, x=(-24, 24), y=(-12, 12),
                    resolution=(384, 192))
disk = grayt.ThinDisk(l0=1.8, r_out=20)

t0 = time.time()
img = grayt.render(bh95, cam, disk)          # no progress output; ~5-10 s
print(f"render: {time.time()-t0:.1f} s for {cam.resolution[0]*cam.resolution[1]} pixels")

ax = img.plot(label="$a=0.95$, $\\ell_0=1.8$")
ax.set_title("Page-Thorne thin disk, edge-on-ish view")
ax.figure.savefig(OUT / "demo_disk_a095.png", dpi=150, bbox_inches="tight")

The bolometric intensity $I_{\rm obs} = g^3 F_{\rm PT}$ shows the
Doppler-boosted approaching side at $x<0$ (the paper's convention) and the
lensed far side of the disk arching over the shadow.

### Inspecting the raw per-pixel maps

`Image` is a plain dataclass of numpy arrays — very friendly for interactive
work: `intensity`, `g` (redshift factor), `r_hit` (Boyer–Lindquist radius of
the disk intersection), `status` (0 escaped / 1 captured / 2 disk / 3 failed,
with module-level `STATUS_*` constants), `herr` (Hamiltonian constraint
error), and the escape direction `theta_inf`, `phi_inf`.

In [ ]:
print("shapes  :", img.intensity.shape, img.g.shape, img.status.shape)
vals, counts = np.unique(img.status, return_counts=True)
print("status  :", dict(zip(vals.tolist(), counts.tolist())),
      " (0=escaped, 1=captured, 2=disk)")
dm = img.status == grayt.STATUS_DISK
print(f"g on disk    : [{img.g[dm].min():.3f}, {img.g[dm].max():.3f}]")
print(f"r_hit on disk: [{img.r_hit[dm].min():.3f}, {img.r_hit[dm].max():.3f}]"
      f"  (ISCO = {bh95.isco:.3f})")
print("max constraint error:", img.herr.max())

fig, axes = plt.subplots(2, 2, figsize=(11, 6), constrained_layout=True)
m0 = axes[0, 0].imshow(np.where(dm, img.g, np.nan).T, origin="lower",
                       extent=img.extent, cmap="RdBu_r", aspect="equal")
axes[0, 0].set_title("redshift factor $g$ (disk pixels)")
fig.colorbar(m0, ax=axes[0, 0])
m1 = axes[0, 1].imshow(np.where(dm, img.r_hit, np.nan).T, origin="lower",
                       extent=img.extent, cmap="viridis", aspect="equal")
axes[0, 1].set_title("$r_{\\rm hit}$ [M]")
fig.colorbar(m1, ax=axes[0, 1])
m2 = axes[1, 0].imshow(img.status.T, origin="lower", extent=img.extent,
                       cmap="tab10", vmin=0, vmax=9, aspect="equal")
axes[1, 0].set_title("status (0 escaped, 1 captured, 2 disk)")
fig.colorbar(m2, ax=axes[1, 0])
axes[1, 1].hist(img.g[dm].ravel(), bins=80, color="tab:purple")
axes[1, 1].axvline(1.0, color="k", lw=0.8, ls="--")
axes[1, 1].set_xlabel("$g$")
axes[1, 1].set_title("histogram of $g$ on the disk")
fig.savefig(OUT / "demo_disk_diagnostics.png", dpi=150, bbox_inches="tight")

Blue-shifted pixels ($g>1$, up to $\sim 1.2$) sit exactly on the
approaching side; `r_hit` bottoms out at the ISCO because `r_in=None`
defaulted there. The maps are self-consistent and easy to slice — this part
of the API is a pleasure.

One nit: typing `img` alone in a cell dumps the full dataclass repr of seven
arrays (~3 kB of text). Harmless, but a custom `__repr__` would be kinder.

## 3. Page–Thorne flux profiles

`grayt.flux_profile(bh, r_out, n)` returns a bare `(r, F)` tuple sampled on
$[r_{\rm ISCO}, r_{\rm out}]$.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for a, color in [(0.0, "tab:blue"), (0.5, "tab:orange"), (0.9, "tab:red")]:
    b = grayt.BlackHole(a=a)
    rs, fs = grayt.flux_profile(b, r_out=20.0)
    print(f"a={a}: profile starts at r={rs[0]:.4f} (ISCO {b.isco:.4f}), "
          f"peak F={fs.max():.3e} at r={rs[np.argmax(fs)]:.3f}")
    ax.plot(rs, fs, color=color, label=f"$a={a}$")
ax.set_xlabel("$r/M$")
ax.set_ylabel("$F(r)$ (Page-Thorne, geometrized)")
ax.set_title("Time-averaged disk flux vs. spin")
ax.legend()
fig.savefig(OUT / "demo_flux_profiles.png", dpi=150, bbox_inches="tight")

Higher spin drags the ISCO inward and raises the peak flux by more than
an order of magnitude between $a=0$ and $a=0.9$ — the standard result, easy
to reproduce in six lines.

## 4. Single geodesics

`grayt.trace` integrates one geodesic from raw initial conditions
`y0 = (t, r, theta, phi, p_r, p_theta)` plus the conserved `p_t`, `p_phi`,
and returns a dict of trajectory columns (nice for plotting, no custom class
to learn). Watch the units seam: here `theta` is in **radians**, while
`Camera.theta` and `orbit_ic(theta0=...)` are in degrees.

`grayt.orbit_ic` builds `(y0, p_t, p_phi)` from conserved $(E, L)$ —
`mass=1` (default) gives a time-like orbit, `mass=0` a photon.

In [ ]:
bh98 = grayt.BlackHole(a=0.98)
esc  = grayt.trace(bh98, [0, 100, np.pi/2, np.pi/2, -1.009327, 1.87],
                   p_t=-0.989953, p_phi=2.000098, lambda_max=220.0)
fall = grayt.trace(bh98, [0, 100, np.pi/2, np.pi/2, -1.009314, 3.75],
                   p_t=-0.989952, p_phi=1.250061, lambda_max=140.0)
print("trace returns:", list(esc))
print("points:", len(esc["r"]), " max |constraint err|:",
      np.abs(esc["herr"]).max())

# time-like bound orbit: turning point at r0 = 25 M around a = 0
bh0 = grayt.BlackHole(a=0.0)
r0, L = 25.0, 4.2
E = np.sqrt((1 - 2/r0)*(1 + L*L/(r0*r0)))      # dr/dlambda = 0 at r0
y0, pt, pphi = grayt.orbit_ic(bh0, r0, E, L)
orb = grayt.trace(bh0, y0, p_t=pt, p_phi=pphi, lambda_max=4000.0,
                  n_max=100_000)

fig, axes = plt.subplots(1, 2, figsize=(11, 5.2), constrained_layout=True)
grayt.plot_orbits_2d([(esc, "photon, escapes"), (fall, "photon, captured")],
                     black_hole=bh98, ax=axes[0],
                     colors=["tab:blue", "tab:red"])
axes[0].set_xlim(-25, 25); axes[0].set_ylim(-25, 25)
axes[0].set_title("photons, $a=0.98$")
grayt.plot_orbits_2d([(orb, f"$L={L}$, $E={E:.3f}$")], black_hole=bh0,
                     ax=axes[1])
axes[1].set_title("time-like rosette, $a=0$")
fig.savefig(OUT / "demo_orbits2d.png", dpi=150, bbox_inches="tight")

`plot_orbits_2d` is unusually flexible for a helper: it accepts trace
dicts, `Ray` objects, or bare `(N, 3)` arrays, optionally `(orbit, label)`
tuples, draws the horizon, and composes onto any axis you hand it.

## 5. Photograph mode: imaging a picture through the spacetime

Place a Lambertian "card" with a loaded image behind the hole and photograph
it with backward ray tracing: one ray per camera pixel, terminated on the
source plane, texture sampled where it lands. Note the scene layer uses
pseudo-Cartesian positions (`center`, `normal`), not Boyer–Lindquist.

In [ ]:
bh9 = grayt.BlackHole(a=0.9)
source = grayt.ImageSource(center=(-150.0, 0.0, 0.0),
                           normal=(1.0, 0.0, 0.0), up=(0.0, 0.0, 1.0),
                           width=90.0, height=68.0,
                           image=str(REPO / "examples/assets/labore_et_constantia.jpg"))
print("texture:", source.image.shape)

ps = grayt.PhysicalSystem(black_hole=bh9, rtol=1e-8, atol=1e-10)
ps.sources.append(source)
sys3 = grayt.System(physical=ps)

cam = grayt.Camera(r=1000.0, theta=90.0, phi=0.0,
                   x=(-45.0, 45.0), y=(-28.0, 28.0), resolution=(450, 280))
t0 = time.time()
photo = sys3.photograph(cam, source, background=0.05)   # ~10 s, silent
print(f"photograph: {time.time()-t0:.1f} s")
vals, counts = np.unique(photo.status, return_counts=True)
print("ray status counts:", dict(zip(vals.tolist(), counts.tolist())),
      "(0 escaped, 1 captured, 2 reached source plane)")

ax = photo.plot(label=f"$a={bh9.a}$")
ax.set_xlabel("$x/M$"); ax.set_ylabel("$y/M$")
ax.set_title("Lambertian card photographed through Kerr spacetime")
ax.figure.savefig(OUT / "demo_photograph.png", dpi=150, bbox_inches="tight")

The direct (distorted) image, the Einstein-ring secondary images and
the shadow all come out at 450×280 in about ten seconds.

## 6. 3D scene with traced rays

`PhysicalSystem.trace_ray(origin, direction)` traces single geodesics from
Cartesian initial data and stores them; `System.visualize3d()` draws horizon,
disk annulus and rays (captured rays black, escaped colored).

In [ ]:
ps2 = grayt.PhysicalSystem(black_hole=grayt.BlackHole(a=0.9),
                           disk=grayt.ThinDisk(r_out=12.0))
observer = np.array([18.0, -14.0, -9.0])
rng = np.random.default_rng(7)
target = -observer/np.linalg.norm(observer)
for _ in range(24):
    ps2.trace_ray(observer, target + rng.normal(scale=0.16, size=3),
                  lambda_max=260.0)
n_cap = sum(r.status == grayt.STATUS_CAPTURED for r in ps2.rays)
print(f"traced {len(ps2.rays)} rays, {n_cap} captured")

scene = grayt.System(physical=ps2)
ax3 = scene.visualize3d(show_surfaces=False, elev=22, azim=-55)
ax3.scatter(*observer, s=80, color="tab:blue", edgecolor="black", zorder=5)
lim = 26
ax3.set_xlim(-lim, lim); ax3.set_ylim(-lim, lim); ax3.set_zlim(-lim, lim)
ax3.set_title("Null geodesics around $a=0.9$ (black = captured)")
ax3.figure.savefig(OUT / "demo_rays3d.png", dpi=150, bbox_inches="tight")

## 7. Edge cases: what happens on bad input?

A quick tour of failure modes, each wrapped in `try/except` so the notebook
still runs top-to-bottom. Verdicts inline.

In [ ]:
# (a) Unphysical spin -- clean, informative ValueError. Good.
try:
    grayt.BlackHole(a=1.5)
except ValueError as e:
    print("BlackHole(a=1.5)      -> ValueError:", e)

# (b) ImageSource without an image -- clean ValueError. Good.
try:
    grayt.ImageSource(center=(-150, 0, 0))
except ValueError as e:
    print("ImageSource(no image) -> ValueError:", e)

# (c) orbit_ic with impossible (E, L) -- clean ValueError. Good.
try:
    grayt.orbit_ic(grayt.BlackHole(a=0.0), r0=10.0, energy=0.5,
                   angular_momentum=4.2)
except ValueError as e:
    print("orbit_ic(bad E,L)     -> ValueError:", e)

In [ ]:
# (d) Disk entirely inside the ISCO: r_in defaults to ISCO (= 6 for a=0),
# which is OUTSIDE r_out=3. No error, no warning -- just a silently
# all-zero image with zero disk hits.
bh0 = grayt.BlackHole(a=0.0)
img_bad = grayt.render(bh0, grayt.Camera(resolution=(96, 48)),
                       grayt.ThinDisk(r_out=3.0))
print("ThinDisk(r_out=3) around a=0 (ISCO=6):")
print("  disk hits:", int((img_bad.status == grayt.STATUS_DISK).sum()),
      "   intensity max:", img_bad.intensity.max(), "  <- silent empty image")

In [ ]:
# (e) Zero-resolution camera: render happily returns (0, 0) arrays;
# the error only surfaces later, deep inside plotting, as an opaque
# numpy reduction failure.
img00 = grayt.render(grayt.BlackHole(a=0.9), grayt.Camera(resolution=(0, 0)),
                     grayt.ThinDisk())
print("render with resolution=(0,0) returned arrays of shape",
      img00.intensity.shape)
try:
    img00.plot()
except ValueError as e:
    print("img00.plot() -> ValueError:", e)

In [ ]:
# (f) The units trap: Camera.theta is in DEGREES. Passing radians is
# accepted silently and gives a face-on view instead of the intended
# edge-on one -- a plausible-looking but wrong image.
img_rad = grayt.render(grayt.BlackHole(a=0.9),
                       grayt.Camera(theta=np.pi/2, resolution=(96, 48)),
                       grayt.ThinDisk(l0=1.8))
print("theta=np.pi/2 interpreted as 1.57 DEGREES (near-polar view);",
      int((img_rad.status == grayt.STATUS_DISK).sum()), "of",
      img_rad.status.size, "pixels hit the disk -- no warning, wrong physics")

# (g) Unknown integrator name -> bare KeyError, valid options not listed.
try:
    grayt.render(grayt.BlackHole(), grayt.Camera(resolution=(8, 8)),
                 method="rk4")
except KeyError as e:
    print("method='rk4' -> KeyError:", e)

### Verdict on error handling

Constructor-level validation (`BlackHole`, `ImageSource`, `orbit_ic`) is
good: early, clear messages. Anything that only becomes inconsistent at
*render* time (disk inside the ISCO, empty camera, degrees-vs-radians) passes
silently, and an unknown `method` leaks a bare `KeyError`. See
`docs/reviews/notebook_user_review.md` for the full ergonomics review.